# 01 · Fine-tune PhoWhisper-small cho môi trường ồn trong xe + đổi sang CTranslate2 int8

Cần **GPU T4**, bật Internet. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Secrets: `HF_TOKEN` (Kaggle: Add-ons → Secrets; Colab: biểu tượng chìa khoá bên trái, bật Notebook access).
Input tuỳ chọn: dataset riêng `xe-noise` gồm các file .wav tiếng ồn tự thu (gió + máy xe máy, điều hoà ô tô).
Trên Colab thì copy thư mục đó lên Drive: `MyDrive/ai-portfolio/viet-copilot/xe-noise/`.
Không có thì dùng tiếng ồn tổng hợp (brown/pink noise + tiếng ù tần số thấp giống động cơ).
Giới hạn ~30 phút GPU: 400 step (batch 32, ~1,1 epoch VIVOS), đánh giá trên 200 câu test.

Dữ liệu giọng: VIVOS (~15h, **CC BY-NC-SA 4.0**: chỉ dùng phi thương mại, model fine-tune cũng chịu điều kiện này).

Câu hỏi: PhoWhisper-small (WER VIVOS 6.33 theo paper) tụt bao nhiêu khi có tiếng ồn xe, và fine-tune với
tiếng ồn trộn ngẫu nhiên kéo lại được bao nhiêu mà không làm hỏng giọng sạch?

In [ ]:
import glob, os, sys
import numpy as np

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/copilot/__init__.py", recursive=True)[0].rsplit("/copilot", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/viet-copilot" if IN_COLAB else os.path.abspath("viet-copilot")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/viet-copilot {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/viet-copilot" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

# datasets 4.x giải mã audio bằng torchcodec (hay lệch phiên bản với torch của Colab) -> dùng bản 3.x
!pip install -q "transformers>=4.44" "datasets>=2.19,<4" jiwer soundfile librosa ctranslate2 accelerate evaluate
from copilot.speech import mix_noise

In [ ]:
from datasets import Audio, load_dataset

vivos = load_dataset("ademax/vivos-vie-speech2text")
print(vivos)
col_audio = [c for c in vivos["train"].column_names if "audio" in c][0]
col_text = [c for c in vivos["train"].column_names if c in ("sentence", "text", "transcription")][0]
vivos = vivos.cast_column(col_audio, Audio(sampling_rate=16000))

## Tiếng ồn

In [ ]:
import soundfile as sf
import librosa

noise_files = glob.glob("/kaggle/input/**/*.wav", recursive=True) + glob.glob(f"{WORK}/xe-noise/**/*.wav", recursive=True)
noises = [librosa.load(f, sr=16000)[0] for f in noise_files]
rng = np.random.default_rng(0)
SR = 16000


def colored(n, power):
    """Ồn có phổ ~ 1/f^power: 0 trắng, 1 hồng (tiếng lốp trên mặt đường), 2 nâu."""
    spec = np.fft.rfft(rng.normal(0, 1, n))
    f = np.fft.rfftfreq(n, 1 / SR)
    spec[1:] /= f[1:] ** (power / 2)
    spec[f < 60] = 0
    x = np.fft.irfft(spec, n)
    return (x / np.abs(x).max()).astype(np.float32)


if not noises:
    # Bản đầu chỉ có brown noise + tiếng ù 45Hz: gần hết năng lượng dưới 200Hz, ngoài dải giọng nói
    # -> WER gần như không đổi kể cả SNR 0 dB, fine-tune chẳng chứng minh được gì. Ồn trong xe thật là dải rộng.
    print("không có ồn thật -> tổng hợp 3 loại: đường (hồng), gió (trắng lọc), máy (nâu + ù)")
    n = SR * 60
    t = np.arange(n) / SR
    road = colored(n, 1.0)
    wind = colored(n, 0.3) * (0.6 + 0.4 * np.sin(2 * np.pi * 0.2 * t))  # gió lúc mạnh lúc yếu
    engine = colored(n, 2.0) + 0.3 * np.sin(2 * np.pi * 45 * t) + 0.15 * np.sin(2 * np.pi * 90 * t)
    noises = [x.astype(np.float32) for x in (road, wind, engine)]
print(len(noises), "đoạn ồn, tổng", round(sum(len(x) for x in noises) / SR), "giây")
# mỗi đoạn ồn: 80% đầu cho train, 20% cuối cho test (model không được nghe đúng đoạn ồn dùng để chấm)
noise_train = [x[: int(len(x) * 0.8)] for x in noises]
noise_test = [x[int(len(x) * 0.8):] for x in noises]

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MID = "vinai/PhoWhisper-small"
proc = WhisperProcessor.from_pretrained(MID, language="vi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MID)
model.generation_config.language = "vi"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

## Đánh giá trước fine-tune: sạch và có ồn (SNR 10 / 5 / 0 dB)

In [ ]:
import jiwer, torch, re

def norm(s):
    return re.sub(r"[^\w\s]", "", s.lower()).strip()

test = vivos["test"].select(range(200))

@torch.no_grad()
def wer_on(m, snr=None, bs=16):
    m.eval().cuda()
    refs, hyps = [], []
    r = np.random.default_rng(123)
    for i in range(0, len(test), bs):
        batch = test[i:i + bs]
        audios = [a["array"].astype(np.float32) for a in batch[col_audio]]
        if snr is not None:
            audios = [mix_noise(a, noise_test[r.integers(len(noise_test))], snr, r) for a in audios]
        feats = proc.feature_extractor(audios, sampling_rate=16000, return_tensors="pt").input_features.cuda().half()
        with torch.autocast("cuda"):
            ids = m.generate(feats, max_new_tokens=128)
        hyps += [norm(t) for t in proc.batch_decode(ids, skip_special_tokens=True)]
        refs += [norm(t) for t in batch[col_text]]
    return round(100 * jiwer.wer(refs, hyps), 2)

before = {f"snr{s}" if s is not None else "sạch": wer_on(model, s) for s in (None, 10, 5, 0)}
before

## Fine-tune: mỗi mẫu train 70% bị trộn ồn với SNR ngẫu nhiên 0-15 dB, 30% giữ sạch
Giữ một phần sạch để không quên giọng sạch. Train ít epoch, lr nhỏ: PhoWhisper đã tốt sẵn.

In [ ]:
from dataclasses import dataclass
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

train = vivos["train"]

def prep(batch, rng=np.random.default_rng(1)):
    a = batch[col_audio]["array"].astype(np.float32)
    if rng.random() < 0.7:
        a = mix_noise(a, noise_train[rng.integers(len(noise_train))], rng.uniform(0, 15), rng)
    batch["input_features"] = proc.feature_extractor(a, sampling_rate=16000).input_features[0]
    batch["labels"] = proc.tokenizer(batch[col_text]).input_ids
    return batch

train = train.map(prep, remove_columns=train.column_names, num_proc=2)

@dataclass
class Collator:
    def __call__(self, feats):
        inp = proc.feature_extractor.pad([{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        lab = proc.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == model.generation_config.decoder_start_token_id).all():
            labels = labels[:, 1:]
        inp["labels"] = labels
        return inp

# 400 step x 32 mẫu ~ 1,1 epoch, ~25 phút trên T4. Không lưu checkpoint giữa chừng (mỗi lần ~3GB lên Drive, chậm)
args = Seq2SeqTrainingArguments(output_dir="/content/ft", per_device_train_batch_size=16, gradient_accumulation_steps=2,
                                learning_rate=1e-5, warmup_steps=50, max_steps=400, fp16=True,
                                gradient_checkpointing=True, logging_steps=50, report_to=[], save_strategy="no")
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train, data_collator=Collator())
trainer.train()

In [ ]:
after = {f"snr{s}" if s is not None else "sạch": wer_on(model, s) for s in (None, 10, 5, 0)}
import pandas as pd
pd.DataFrame({"PhoWhisper-small gốc": before, "fine-tune có ồn": after})

## Đổi sang CTranslate2 int8 (chạy bằng faster-whisper trên laptop) + đẩy lên HF Hub

In [ ]:
import json
model.save_pretrained("/content/phowhisper-noisy"); proc.save_pretrained("/content/phowhisper-noisy")
# bản CT2 int8 (~250MB) để trên Drive: laptop tải về chạy bằng faster-whisper
!ct2-transformers-converter --model /content/phowhisper-noisy --output_dir models/phowhisper-noisy-ct2-int8 \
    --quantization int8 --copy_files tokenizer.json preprocessor_config.json --force
!du -sh models/phowhisper-noisy-ct2-int8
json.dump({"before": before, "after": after}, open("wer.json", "w"), ensure_ascii=False, indent=1)

token = secret("HF_TOKEN")
if not token:
    print("thiếu secret HF_TOKEN -> chưa đẩy lên HF Hub (model vẫn nằm trên Drive)")
else:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    REPO = f"{api.whoami()['name']}/phowhisper-small-noisy-ct2-int8"
    api.create_repo(REPO, exist_ok=True)
    api.upload_folder(folder_path="models/phowhisper-noisy-ct2-int8", repo_id=REPO)
    api.upload_file(path_or_fileobj="wer.json", path_in_repo="wer.json", repo_id=REPO)